# Merging Property and Applicant Data with Price Completion

## Overview
This script processes and merges property and applicant datasets for Chapter 40B housing projects. It performs the following key tasks:

1. **Load datasets**  
   - Property dataset containing price and resale information.  
   - Applicant dataset containing application details.

2. **Standardize columns for merging**  
   - Cleans town, address, and unit columns by stripping whitespace and converting to lowercase for consistent matching.

3. **Merge applicants onto properties**  
   - Left join ensures all properties are retained, even if they have no applicants.

4. **Add missing Maximum Resale Prices**  
   - Merges additional price data to fill any missing "Maximum Resale Price" values.  

5. **Save the final dataset**  
   - Outputs a CSV with merged applicant data and completed price information.

This ensures a comprehensive property dataset with applicants and complete resale price information, ready for further analysis.


In [ ]:
import pandas as pd

# ===============================
# Step 1: Load datasets
# ===============================
# Property dataset with price & resale info
df_properties = pd.read_csv("/content/merged_dataset_price&resale_Sept25.csv")

# Applicant dataset for Chapter 40B properties
df_applicants = pd.read_csv("/content/CHAPA_Chapter-40B_Application-Data_2021-2025_merged_v0.3.csv")

# ===============================
# Step 2: Standardize columns for merging
# ===============================
# Function to clean text columns (strip spaces, convert to lowercase)
def clean_text(series):
    return series.astype(str).str.strip().str.lower()

# Clean property dataset columns
df_properties['Town_clean'] = clean_text(df_properties['Town'])
df_properties['Address_clean'] = clean_text(df_properties['Address'])
df_properties['Unit_clean'] = clean_text(df_properties['Unit Number'])

# Clean applicant dataset columns
df_applicants['Town_clean'] = clean_text(df_applicants['property_town_city'])
df_applicants['Address_clean'] = clean_text(df_applicants['property_street_address'])
df_applicants['Unit_clean'] = clean_text(df_applicants['property_unit'])

# ===============================
# Step 3: Merge applicants onto properties
# ===============================
# Left join ensures all properties are kept even if there are no applicants
merged = pd.merge(
    df_properties,
    df_applicants,
    on=['Town_clean', 'Address_clean', 'Unit_clean'],
    how='left',
    suffixes=('', '_applicant')
)

# Drop the temporary cleaned columns
merged = merged.drop(columns=['Town_clean', 'Address_clean', 'Unit_clean'])

# Save intermediate merge
merged.to_csv("merged_properties_with_applicants.csv", index=False)
print("✅ Merge complete. File saved as merged_properties_with_applicants.csv")

# ===============================
# Step 4: Add missing Maximum Resale Prices
# ===============================
# Load the merged dataset and new price data
merged_final = pd.read_csv("merged_properties_with_applicants.csv")
df_prices = pd.read_csv("/content/Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed (1).csv")

# Rename column for clarity
df_prices = df_prices.rename(columns={"Price": "Maximum Resale Price"})

# Define keys to merge on
merge_keys = ["Town", "Address", "Unit Number"]

# Merge new prices into the dataset
merged_updated = pd.merge(
    merged_final,
    df_prices[merge_keys + ["Maximum Resale Price"]],
    on=merge_keys,
    how="left",
    suffixes=("", "_new")
)

# Fill missing prices with values from new dataset
merged_updated["Maximum Resale Price"] = merged_updated["Maximum Resale Price"].fillna(
    merged_updated["Maximum Resale Price_new"]
)

# Drop the helper column
merged_updated = merged_updated.drop(columns=["Maximum Resale Price_new"])

# Save the final dataset
merged_updated.to_csv("new_merged_dataset_filled.csv", index=False)
print("✅ Missing prices filled and saved as new_merged_dataset_filled.csv")
